# 📘 Logistic Regression – Study Notes
---

## 1. Why Not Linear Regression for Classification?

- Linear regression predicts **continuous numbers** — not suitable for categories
- Mapping categories as 0, 1, 2... implies a false ordering (e.g., Neutral is not "between" Satisfied and Not Satisfied)
- Predictions can go **below 0 or above 1** — doesn't make sense for probability
- It can work *okay* for binary classification (2 classes), but **fails for multi-class**
- Solution → use **Logistic Regression** for classification

## 2. What is Logistic Regression?

- A **classification** algorithm (despite having "regression" in the name)
- Predicts the **probability** that a data point belongs to a class
- If probability > 50% → assign to that class, else → assign to the other class
- Acts as a **binary classifier** by default

**Core idea:**
- Linear regression gives: $y = \beta_0 + \beta_1 x$ → can go beyond 0–1
- Logistic regression wraps this in a **Sigmoid function** → output always between 0 and 1

## 3. Sigmoid Function

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Why Sigmoid?**
1. Output is always between **0 and 1** → perfect for probability
2. Its **derivative is easy to compute** → helpful in gradient descent
3. Introduces **non-linearity** into the model

The output of sigmoid is the **predicted probability** of belonging to the positive class.

## 4. Logit Function

- Taking the log of the odds gives us the **logit function**
- The logit is **linear in terms of features x** → that's the regression part
- This is how logistic regression connects to linear regression internally

$$\text{logit}(p) = \ln\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x$$

## 5. Cost Function & How the Model Learns

- We **cannot use RSS** (from linear regression) here — it gives a non-convex curve with many local minima
- Instead, we use **Cross-Entropy Loss** (log loss):

$$E_{in}(\mathbf{w}) = \frac{1}{N} \sum_{n=1}^{N} \ln\left(1 + e^{-y_n \mathbf{w}^T \mathbf{x}_n}\right)$$

- This is **convex** → only one minimum → guaranteed to find the global minimum
- Minimized using **Batch Gradient Descent**

**Weight update rule:**
$$\mathbf{w}_{i+1} = \mathbf{w}_i + \eta \cdot \frac{1}{N} \sum_{n=1}^{N} \frac{y_n \mathbf{x}_n}{1 + e^{y_n \mathbf{w}_i^T \mathbf{x}_n}}$$

- `η` (eta) = learning rate
- Weights keep updating until minimum error is reached

## 6. Multiple Logistic Regression

- Same as logistic regression but with **multiple features**
- Coefficients are learned the same way (via gradient descent on the cost function)
- Check for **multicollinearity** between features just like in linear regression (use VIF)

## 7. Multinomial Logistic Regression (More than 2 classes)

**Approach 1 – Softmax (One-vs-Rest):**
- Train a separate model for each class
- Each model gives a probability for its class
- Predict the class with the **highest probability**
- All probabilities sum to 1

**Approach 2 – One vs One:**
- Train a model for every pair of classes (A vs B, A vs C, B vs C)
- Majority vote decides the final class

**Approach 3 – One vs Many:**
- Each class is trained against all remaining classes

> Note: For more than 2 classes, other algorithms (like Decision Trees, SVM) are often preferred over logistic regression.

## 8. Confusion Matrix

A table that shows how well the model predicted vs actual results:

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actually Positive** | TP (True Positive) | FN (False Negative) |
| **Actually Negative** | FP (False Positive) | TN (True Negative) |

- **TP** – Correctly predicted positive ✅
- **TN** – Correctly predicted negative ✅
- **FP** – Predicted positive, but actually negative ❌ (Type I Error)
- **FN** – Predicted negative, but actually positive ❌ (Type II Error)

## 9. Evaluation Metrics

### Accuracy
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
- % of total correct predictions
- ⚠️ Misleading when classes are imbalanced (e.g., 95% not-cancer, 5% cancer)

### Recall (Sensitivity)
$$\text{Recall} = \frac{TP}{TP + FN}$$
- Of all actual positives, how many did we catch?
- Use when **missing a positive is costly** (e.g., cancer detection)

### Precision
$$\text{Precision} = \frac{TP}{TP + FP}$$
- Of all predicted positives, how many were actually positive?
- Use when **false alarms are costly** (e.g., wrongly accusing someone)

### F1 Score
$$F1 = \frac{2 \times Precision \times Recall}{Precision + Recall}$$
- **Harmonic mean** of Precision and Recall
- Best metric when you need a balance of both
- Use when you **can't sacrifice** either precision or recall

### Specificity (True Negative Rate)
$$\text{Specificity} = \frac{TN}{TN + FP}$$
- Of all actual negatives, how many did we correctly identify?

## 10. Precision vs Recall Tradeoff

- As Recall increases → Precision decreases (and vice versa)
- You **cannot maximize both** at the same time
- The right choice depends on the **business requirement**:

| Scenario | Prioritize |
|---|---|
| Cancer detection | Recall (don't miss positives) |
| Fraud detection | Recall |
| Spam filter | Precision (don't mark real emails as spam) |
| Innocence prediction | Precision |

- When unsure → use **F1 Score**

## 11. Threshold & ROC Curve

**Threshold:**
- Logistic regression outputs probabilities (0 to 1)
- We pick a **cutoff** (default = 0.5): above → positive, below → negative
- Changing the threshold changes the TP, FP, FN, TN counts

**ROC Curve (Receiver Operating Characteristic):**
- Plots **True Positive Rate (Recall)** vs **False Positive Rate** at different thresholds
- Each point on the curve = one threshold setting
- The **green dotted diagonal line** = random guessing (TPR = FPR)
- **Best point** = top-left corner (high TPR, low FPR)
- Helps answer: **which threshold should I use?**

## 12. AUC – Area Under the ROC Curve

- When comparing **multiple models**, use AUC to pick the best one
- **Higher AUC = better model**
- AUC = 1.0 → perfect model
- AUC = 0.5 → random guessing (useless model)
- AUC < 0.5 → worse than random

> Example: If Logistic Regression gives AUC = 0.85 and SVM gives AUC = 0.91, choose SVM.

## 13. Practical Implementation – Diabetes Dataset

**Steps followed in the notebook:**

1. **Load data** – `diabetes.csv` with features like Glucose, BMI, Age, etc.
2. **Handle 0 values** – Columns like Glucose, BMI can't be 0 → replace with column mean
3. **Handle outliers** – Remove top 1–5% extreme values using quantile method
4. **Scale features** – Use `StandardScaler` (needed before logistic regression)
5. **Check multicollinearity** – VIF for all features → all < 5 (good)
6. **Train/test split** – 75% train, 25% test
7. **Fit model** – `LogisticRegression()` from sklearn
8. **Evaluate** – Accuracy, Precision, Recall, F1, AUC, ROC curve
9. **Save model** – Using `pickle` for later use

**Key code snippets:**
```python
from sklearn.linear_model import LogisticRegression
log_reg = LogisticRegression()
log_reg.fit(x_train, y_train)
y_pred = log_reg.predict(x_test)
```

## 14. Advantages & Disadvantages

### ✅ Advantages
- Simple and easy to implement
- Output is a probability → more informative than just a label
- Shows relationship between features and target
- Works very well with **linearly separable** data

### ❌ Disadvantages
- Doesn't work well when data is **not linearly separable**
- Weaker than other classifiers (SVM, Random Forest, etc.)
- Multi-class problems are easier to handle with other algorithms
- Can only predict **categorical outcomes**

## 15. Quick Summary

| Concept | Key Point |
|---|---|
| Logistic Regression | Classification, not regression |
| Sigmoid Function | Converts any value to probability (0–1) |
| Cost Function | Cross-entropy loss (convex) |
| Learning | Gradient Descent on cost function |
| Threshold | Default 0.5; changing it affects TP/FP/FN/TN |
| Confusion Matrix | TP, TN, FP, FN breakdown |
| Accuracy | Overall correct predictions |
| Recall | Catches all actual positives (minimize FN) |
| Precision | Avoids false alarms (minimize FP) |
| F1 Score | Balance between Precision and Recall |
| ROC Curve | TPR vs FPR at all thresholds |
| AUC | Compares models; higher = better |
| VIF | Check multicollinearity (keep < 5) |
| Multinomial | One-vs-Rest or Softmax for 3+ classes |